## Angewandtes Quantencomputing Laborpraktikum

### Praktikum 7: Grover-Algorithmus

Der **Grover-Algorithmus** ist ein zentraler Quantenalgorithmus zur Lösung des Problems der *unstrukturierten Suche*.  
Er ermöglicht es, ein gesuchtes Element in einer Menge von $N$ Möglichkeiten mit nur $O(\sqrt{N})$ Orakelabfragen zu finden. Die klassische unstrukturierte Suche benötigt dagegen $O(N)$ Abfragen. Grover erzielt damit einen quadratischen Vorteil bei der Anzahl der Abfragen. Die Gesamtlaufzeit hängt zusätzlich vom Aufwand für die Umsetzung des Orakels und der übrigen Schaltung ab.

Die Grundidee: Statt jedes Element einzeln zu prüfen, werden alle Möglichkeiten gleichzeitig in einer **Superposition** betrachtet.  
Durch das gezielte Zusammenspiel von **Orakel** und **Amplitudenverstärkung** werden die richtigen Lösungen schrittweise „herausgefiltert“.

**Drei Bausteine:**

1. **Superposition:** Erzeugung einer gleichverteilten Überlagerung aller möglichen Zustände.  
2. **Orakel:** Markierung der Lösungszustände durch Umkehr ihres Amplituden-Vorzeichens.  
3. **Amplitudenverstärkung:** Verstärkung der markierten Zustände, sodass ihre Messwahrscheinlichkeit steigt.

Nach einigen Iterationen ist die Wahrscheinlichkeit sehr hoch, bei einer Messung die gesuchte Lösung zu erhalten.

In diesem Praktikum implementieren und analysieren wir die Anwendung des Grover-Algorithmus auf ein einfaches Beispiel. 
Wir suchen unter den Quantenzuständen genau den Zustand, bei dem alle Qubits den Wert 1 haben. 

Außerdem behandeln wir die **Amplitudenverstärkung** in diesem Praktikum als **Black Box**: Wir verwenden eine vorgegebene Schaltung, ohne ihre Herleitung im Detail zu diskutieren.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qc_lecture_tools.statevector import sv_state

### Vorbereitung

In diesem Notebook verwenden wir durchgehend:

- `sv_state(quantum_circuit)`, um den Quantenzustand zu untersuchen
- `quantum_circuit1.compose(quantum_circuit2, inplace=True)`, um Teilschaltungen aneinanderzuhängen
- `draw("mpl")`, um Schaltungen grafisch darzustellen

Wir suchen (mit Ausnahme der Bonusaufgabe) immer genau **einen** Zielzustand, nämlich den Zustand mit lauter Einsen, also z. B. $\ket{111}$ für 3 Qubits.


### Aufgabe 1: Gleichverteilte Superposition für 3 Daten-Qubits

Zunächst bringen wir alle drei Daten-Qubits in eine gleichverteilte Superposition.  
Dazu wenden wir auf jedes Daten-Qubit ein Hadamard-Gatter an.

Erzeugen Sie die gleichverteilte Superposition für 3 Qubits und untersuchen Sie den Zustand mit `sv_state(quantum_circuit)`.


In [ ]:
num_qubits = 3
qc_superposition = QuantumCircuit(num_qubits)

# TODO Aufgabe 1


### Aufgabe 2: Einfaches Orakel für den Zustand $\ket{111}$

Nun bauen wir ein Orakel, das genau den Zustand $\ket{111}$ markiert. Markieren bedeutet hier: Die Amplitude dieses Zustands erhält ein negatives Vorzeichen.

Dies können wir prinzipiell mit einem mehrfach kontrollierten $Z$-Gatter umsetzen. 

Leider ist ein mehrfach kontrolliertes $Z$-Gatter in Qiskit nicht direkt verfügbar. 
Wir konstruieren es hier aus bekannten Bausteinen: einem mehrfach kontrollierten $X$-Gatter und Hadamard-Gattern. Dabei nutzen wir die Beziehung $HXH = Z$ und gehen wie folgt vor:

- Ein mehrfach kontrolliertes $X$-Gatter (`mcx`)
- Hadamard-Gatter auf dem Ziel-Qubit vor und nach dem mehrfach kontrollierten $X$-Gatter
- Dadurch entsteht effektiv ein mehrfach kontrolliertes $Z$-Gatter

**Aufgabe 2.1:** Bauen Sie ein Orakel für 3 Qubits, das den Zustand $\ket{111}$ markieren kann. Die Funktionalität können Sie in Aufgabe 2.2 überprüfen. 


In [ ]:
num_qubits = 3
qc_oracle = QuantumCircuit(num_qubits)

# TODO Aufgabe 2.1


**Aufgabe 2.2:** Wenden Sie das Orakel auf die gleichverteilte Superposition an und untersuchen Sie den Zustand mit `sv_state`.
Nutzen Sie dafür die `compose`-Funktion. Was beobachten Sie hier? Was bedeutet das für die Messwahrscheinlichkeiten?

**Hinweis:** Mit der folgenden Qiskit-Funktion kann ein zweiter Schaltkreis (`quantum_circuit2`) an einen bestehenden (`quantum_circuit1`) angehängt werden:

```python
quantum_circuit1.compose(quantum_circuit2, inplace=True)
```

In [ ]:
# TODO Aufgabe 2.2


### Aufgabe 3: Amplitudenverstärkung

Als nächsten Schritt verwenden wir die Amplitudenverstärkung. In diesem Praktikum behandeln wir sie als **vorgegebene Black Box**.

Wir benutzen hier die bereits implementierte Funktion `amplitude_amplification`.
Sie erzeugt den Schaltkreis für die Amplitudenverstärkung für eine gegebene Anzahl von Qubits.

Wichtig ist hier nur: Nach dem Orakel verstärkt die Amplitudenverstärkung die Amplitude des markierten Zustands.

**Aufgabe 3.1:** Führen Sie die folgende Zelle aus und betrachten Sie den Schaltkreis. Erkennen Sie einige Komponenten wieder?


In [ ]:
from qc_lecture_tools.grover import amplitude_amplification

num_qubits = 3
qc_amplification = amplitude_amplification(num_qubits, 0)
display(qc_amplification.draw("mpl"))

**Aufgabe 3.2:** Wenden Sie die Amplitudenverstärkung auf den Zustand nach dem Orakel an und untersuchen Sie das Ergebnis.


In [ ]:
# TODO Aufgabe 3.2

### Aufgabe 4: Implementierung des Grover-Algorithmus

Der Grover-Algorithmus beginnt mit der Erzeugung der gleichverteilten Superposition.

Anschließend werden Orakel und Amplitudenverstärkung in mehreren Grover-Schritten angewendet.  

Ein Grover-Schritt besteht aus:

1. Orakel  
2. Amplitudenverstärkung

Für den Grover-Algorithmus sind in der Regel mehrere Grover-Schritte notwendig. Orakel und Amplitudenverstärkung werden also mehrfach abwechselnd angewendet. 

Wir untersuchen nun, was mit den Wahrscheinlichkeiten passiert, wenn wir mehrere Grover-Schritte durchführen. 

**Aufgabe 4.1:** Wenden Sie Orakel und Amplitudenverstärkung ein zweites Mal an. Wie sieht nun die Wahrscheinlichkeit aus, den korrekten Zustand zu messen?


In [ ]:
# TODO Aufgabe 4.1

**Aufgabe 4.2:** Führen Sie einen dritten Grover-Schritt aus. Vergleichen Sie die Ausgabe mit den vorherigen Zellen. Was ist jetzt passiert?


In [ ]:
# TODO Aufgabe 4.2

### Aufgabe 5: Wie sieht das Ganze für 4 oder 5 Qubits aus?

Je mehr Qubits wir haben, desto mehr Basiszustände gibt es insgesamt:

- bei 4 Qubits: $2^4 = 16$
- bei 5 Qubits: $2^5 = 32$

Wir suchen aber weiterhin nur **einen** Zustand, in diesem Beispiel den Zustand $\ket{1\ldots1}$.
Darum brauchen wir bei mehr Qubits auch **mehr Grover-Schritte**.

Da unser Quantenzustand nun sehr viele Zustände enthält, verwenden wir für die Ausgabe eine kompaktere Darstellung.

Nutzen Sie dafür die Funktion 
`print_grover_states(quantum_circuit, num_qubits)`


**Aufgabe 5.1:** Implementieren Sie nun den Grover-Algorithmus für 4 Qubits.
Nutzen Sie den Code aus den vorherigen Aufgaben und passen Sie die Schaltkreise an die neue Qubit-Zahl an.
Wie viele Grover-Schritte sind nun notwendig, um die maximale Wahrscheinlichkeit für den Zustand $\ket{1111}$ zu erreichen?


In [ ]:
from qc_lecture_tools.grover import print_grover_states

num_qubits = 4

qc_superposition_4 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.1: Superposition vorbereiten

qc_oracle_4 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.1: Orakel implementieren

qc_amplitude_amplification_4 = amplitude_amplification(num_qubits, 0)

qc_grover_4 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.1: Teilschaltungen zusammensetzen und Grover-Schritte untersuchen

print_grover_states(qc_grover_4, num_qubits)


**Aufgabe 5.2:** Wiederholen Sie das Beispiel für **5 Daten-Qubits** und den Zielzustand $\ket{11111}$.


In [ ]:
num_qubits = 5

qc_superposition_5 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.2: Superposition vorbereiten

qc_oracle_5 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.2: Orakel implementieren

qc_amplification_5 = amplitude_amplification(num_qubits, 0)

qc_grover_5 = QuantumCircuit(num_qubits)
# TODO Aufgabe 5.2: Teilschaltungen zusammensetzen und Grover-Schritte untersuchen

print_grover_states(qc_grover_5, num_qubits)


### Aufgabe 6: Compute–Uncompute und Ancilla-Qubits

Für die Konstruktion des Orakels haben wir bisher ausgenutzt, dass wir den gesuchten Zustand bereits kennen, nämlich $\ket{1\ldots1}$. 
Nun implementieren wir ein Orakel über eine logische Abfrage, bei der wir die Lösung nicht direkt fest vorgeben. 

Wie in Praktikum 6 (Logische Abfragen) brauchen wir dafür zusätzliche Qubits – **Ancilla-Qubits** –, in denen wir Zwischenergebnisse der logischen Abfrage speichern können. 

Wir unterscheiden im Folgenden:
- **Daten-Qubits:** Qubits, deren Zustände wir mit dem Grover-Algorithmus durchsuchen wollen. 
- **Ancilla-Qubits:** zusätzliche Qubits, die wir für die Implementierung des Orakels benötigen. 

Wir können das Ergebnis der logischen Abfrage in einem zusätzlichen Ancilla-Qubit speichern und anschließend darauf ein $Z$-Gatter anwenden. 

Nach der Berechnung der logischen Abfrage sind die Ancilla-Qubits im Allgemeinen mit den Daten-Qubits verschränkt. 
Damit die anschließende Amplitudenverstärkung auf den Daten-Qubits wie vorgesehen funktioniert, müssen die Ancilla-Qubits bereits **vor diesem Schritt** wieder in ihren Ausgangszustand zurückgeführt werden. Dabei bleibt die durch das $Z$-Gatter erzeugte Phasenmarkierung auf den Daten-Qubits erhalten.

Dies geschieht im **Uncompute**-Schritt. Danach sind die Ancilla-Qubits wieder von den Daten-Qubits getrennt und können auch bei weiteren Orakelaufrufen erneut verwendet werden.

Die Idee beim Compute–Uncompute-Orakel ist:

1. Die Bedingung „alle Daten-Qubits sind 1“ wird auf das Ancilla-Qubit **berechnet**.  
2. Auf dem Ancilla-Qubit wird ein $Z$-Gatter angewendet.  
3. Danach wird der Rechenschritt wieder **rückgängig gemacht**.

Dadurch befindet sich das Ancilla-Qubit am Ende wieder im Ausgangszustand.

Wichtig in diesem Notebook:  
Die **Superposition liegt nur auf den Daten-Qubits**. Das Ancilla-Qubit bleibt zu Beginn im Zustand $\ket{0}$.


**Aufgabe 6.1:** Starten wir wieder mit der gleichverteilten Superposition. Hier wenden wir das Hadamard-Gatter nur auf die Daten-Qubits an, nicht auf das Ancilla-Qubit.


In [ ]:
num_qubits = 3
num_ancilla_qubits = 1
num_qubits = num_qubits + num_ancilla_qubits

qc_superposition = QuantumCircuit(num_qubits)

# TODO Aufgabe 6.1


**Aufgabe 6.2:** Wir bauen nun eine Compute–Uncompute-Implementierung für ein Orakel, das den Zustand $\ket{111}$ findet. 
Dazu nutzen wir das mehrfach kontrollierte $X$-Gatter `mcx`, um das Ancilla-Qubit von 0 auf 1 zu setzen, falls alle drei Daten-Qubits den Wert 1 haben. 
Anschließend wenden wir ein $Z$-Gatter auf das Ancilla-Qubit an. Im letzten Schritt muss das Ancilla-Qubit wieder auf 0 zurückgesetzt werden. Wie kann das umgesetzt werden?

Implementieren Sie die Compute–Uncompute-Version des Orakels. 

Testen Sie Ihre Implementierung, indem Sie das Orakel auf die gleichverteilte Superposition anwenden und den Zustand analysieren. 
Das Vorzeichen des Zustands $\ket{111}$ sollte negativ sein; das zusätzliche 4. Qubit (das Ancilla-Qubit) sollte wieder auf 0 stehen.


In [ ]:
num_qubits = 3
num_ancilla_qubits = 1
num_qubits = num_qubits + num_ancilla_qubits

qc_oracle_compute_uncompute = QuantumCircuit(num_qubits)
# TODO Aufgabe 6.2: Compute-Uncompute-Orakel implementieren

qc_after_oracle_compute_uncompute = QuantumCircuit(num_qubits)
# TODO Aufgabe 6.2: Orakel auf die gleichverteilte Superposition anwenden und den Zustand untersuchen


**Aufgabe 6.3:** Ein kompletter Grover-Schritt mit Compute–Uncompute-Orakel

Wir implementieren nun wieder den Grover-Algorithmus. Nutzen Sie wieder die vorimplementierte Version der Amplitudenverstärkung:

> `amplitude_amplification(3, 1)`

Im ersten Argument der Funktion steht die Anzahl der Daten-Qubits, im zweiten die Anzahl der Ancilla-Qubits. 

Analysieren Sie den Zustand nach einem und nach zwei Grover-Schritten.


In [ ]:
qc_grover = QuantumCircuit(num_qubits)

# TODO Aufgabe 6.3


### Bonusaufgabe: Neues Orakel

**Aufgabe 7.1:**
Nun ändern wir die gesuchten Zustände. Wir suchen alle Zustände, bei denen **genau zwei** Qubits im Zustand 1 sind. 
Dies kann mit drei zweifach kontrollierten und einem dreifach kontrollierten $X$-Gatter umgesetzt werden. 
Die Abfrage soll auf 3 Daten-Qubits funktionieren.

Implementieren Sie dafür das Compute–Uncompute-Orakel. 

Testen Sie Ihren Schaltkreis wie zuvor, indem Sie ihn auf die gleichverteilte Superposition anwenden.


In [ ]:
qc_compute_uncompute_7 = QuantumCircuit(num_qubits)

# TODO Aufgabe 7.1: Compute-Uncompute-Orakel implementieren


In [ ]:
qc_grover = QuantumCircuit(num_qubits)

# TODO Aufgabe 7.1: Orakel auf die gleichverteilte Superposition anwenden und den Zustand untersuchen


**Aufgabe 7.2:** Implementieren Sie nun wieder den Grover-Algorithmus. Wie sehen hier die Wahrscheinlichkeiten aus?
Analysieren Sie das Ergebnis des Schaltkreises nach einem Grover-Schritt mit `print_grover_states(quantum_circuit, 3)`.


In [ ]:
qc_grover = QuantumCircuit(num_qubits)

# TODO Aufgabe 7.2
